In [77]:
!pip install pandas matplotlib seaborn



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [78]:
import os
print("Current directory:", os.getcwd())
print("Files here:", os.listdir())


Current directory: c:\Users\camil\Desktop\Python project Egypt\code
Files here: ['01_data_cleaning - Copia.ipynb', '01_data_cleaning.ipynb']


First, I want to see how many elements my first dataset has (ACLED_egypt2011), and check that the timeframe is correct.

In [79]:
from pathlib import Path
import pandas as pd

# Works whether the notebook runs from project root or the code/ folder.
data_path = Path("..") / "raw_data" / "ACLED_egypt2011.csv"
if not data_path.exists():
    data_path = Path("raw_data") / "ACLED_egypt2011.csv"

df_egypt2011 = pd.read_csv(data_path)
print(f"Loaded {len(df_egypt2011):,} rows and {df_egypt2011.shape[1]} columns from {data_path}")
df_egypt2011.head()

Loaded 2,607 rows and 32 columns from ..\raw_data\ACLED_egypt2011.csv


,event_id_cnty,event_date,year,time_precision,disorder_type,event_type,sub_event_type,actor1,assoc_actor_1,inter1,...,latitude,longitude,geo_precision,source,source_scale,notes,fatalities,tags,timestamp,population_best
0,EGY2776,2013-07-02,2013,1,Demonstrations,Protests,Peaceful protest,Protesters (Egypt),Muslim Brotherhood,Protesters,...,31.1143,30.9401,1,Egypt Independent,National,"In Kafr al-Sheikh, Nile Delta governorate, hun...",0,NaN,1618528752,NaN
1,EGY2748,2013-07-01,2013,1,Demonstrations,Protests,Peaceful protest,Protesters (Egypt),NaN,Protesters,...,29.3099,30.8418,1,Egypt Independent,National,"In al-Massala Sqaure, dozens of opponents prot...",0,NaN,1618528749,NaN
2,EGY2749,2013-07-01,2013,1,Demonstrations,Protests,Peaceful protest,Protesters (Egypt),Muslim Brotherhood,Protesters,...,30.0310,31.1111,1,Egypt Independent,National,Dozens of members of the Muslim Brotherhood be...,0,NaN,1618528749,NaN
3,EGY2665,2013-06-19,2013,1,Demonstrations,Protests,Peaceful protest,Protesters (Egypt),NaN,Protesters,...,30.5526,31.0090,1,Egypt Independent,National,Protesters continued their sit-in for a third ...,0,NaN,1618528740,NaN
4,EGY2659,2013-06-18,2013,1,Demonstrations,Riots,Violent demonstration,Rioters (Egypt),NaN,Rioters,...,30.9761,31.1669,1,Egypt Independent,National,"In a separate incident, Kafr al-Sheikh Governo...",0,NaN,1618528739,NaN


In [80]:
# Time window for protest events in the cleaned dataset
protests_dates = pd.to_datetime(protests["event_date"], errors="coerce")

start_date = protests_dates.min()
end_date = protests_dates.max()

print(f"Protest events time window: {start_date.date()} to {end_date.date()}")
print(f"Total span: {(end_date - start_date).days} days")

Protest events time window: 2011-01-25 to 2013-08-14
Total span: 932 days


Now, since I want to focus on domestic protests against the government, I decided to filter out protests events related to the Israeli-Palestinian conflict. Therefore, I will exclude from the dataset the protests occurred that include the words 'israel', 'israeli', 'palestine, 'palestinian'.

In [81]:
import re

keywords = ["israel", "israeli", "palestine", "palestinian"]
document_text = " ".join(df_egypt2011.fillna("").astype(str).to_numpy().ravel()).lower()

whole_word_counts = {
    kw: len(re.findall(rf"\b{re.escape(kw)}\b", document_text))
    for kw in keywords
}

print("Whole-word occurrences:")
for kw, count in whole_word_counts.items():
    print(f"{kw}: {count}")

print(f"Total whole-word matches: {sum(whole_word_counts.values())}")

Whole-word occurrences:
israel: 8
israeli: 51
palestine: 7
palestinian: 10
Total whole-word matches: 76


I also remove mob violence events

In [82]:
import re

exclude_keywords = ["israel", "israeli", "palestine", "palestinian"]
exclude_pattern = r"\b(?:" + "|".join(map(re.escape, exclude_keywords)) + r")\b"

# Combine each row into one string, then mark rows that contain any exclusion keyword.
row_text = df_egypt2011.fillna("").astype(str).agg(" ".join, axis=1)
mask_excluded = row_text.str.contains(exclude_pattern, case=False, na=False, regex=True)

# Also remove mob violence at cleaning stage (before analysis/plots).
mask_mob_egypt2011 = (
    df_egypt2011["sub_event_type"]
    .astype(str)
    .str.strip()
    .str.lower()
    .str.contains(r"\bmob violence\b", na=False, regex=True)
 )

mask_remove_egypt2011 = mask_excluded | mask_mob_egypt2011
df_egypt2011_clean = df_egypt2011.loc[~mask_remove_egypt2011].copy()

removed_egypt2011_keywords = mask_excluded.sum()
removed_egypt2011_mob_only = ((~mask_excluded) & mask_mob_egypt2011).sum()

print(f"Original events: {len(df_egypt2011):,}")
print(f"Removed events (keywords): {removed_egypt2011_keywords:,}")
print(f"Removed events (mob violence): {removed_egypt2011_mob_only:,}")
print(f"Removed events (total): {mask_remove_egypt2011.sum():,}")
print(f"Remaining events: {len(df_egypt2011_clean):,}")

df_egypt2011_clean.head()

Original events: 2,607
Removed events (keywords): 43
Removed events (mob violence): 241
Removed events (total): 284
Remaining events: 2,323


,event_id_cnty,event_date,year,time_precision,disorder_type,event_type,sub_event_type,actor1,assoc_actor_1,inter1,...,latitude,longitude,geo_precision,source,source_scale,notes,fatalities,tags,timestamp,population_best
0,EGY2776,2013-07-02,2013,1,Demonstrations,Protests,Peaceful protest,Protesters (Egypt),Muslim Brotherhood,Protesters,...,31.1143,30.9401,1,Egypt Independent,National,"In Kafr al-Sheikh, Nile Delta governorate, hun...",0,NaN,1618528752,NaN
1,EGY2748,2013-07-01,2013,1,Demonstrations,Protests,Peaceful protest,Protesters (Egypt),NaN,Protesters,...,29.3099,30.8418,1,Egypt Independent,National,"In al-Massala Sqaure, dozens of opponents prot...",0,NaN,1618528749,NaN
2,EGY2749,2013-07-01,2013,1,Demonstrations,Protests,Peaceful protest,Protesters (Egypt),Muslim Brotherhood,Protesters,...,30.0310,31.1111,1,Egypt Independent,National,Dozens of members of the Muslim Brotherhood be...,0,NaN,1618528749,NaN
3,EGY2665,2013-06-19,2013,1,Demonstrations,Protests,Peaceful protest,Protesters (Egypt),NaN,Protesters,...,30.5526,31.0090,1,Egypt Independent,National,Protesters continued their sit-in for a third ...,0,NaN,1618528740,NaN
4,EGY2659,2013-06-18,2013,1,Demonstrations,Riots,Violent demonstration,Rioters (Egypt),NaN,Rioters,...,30.9761,31.1669,1,Egypt Independent,National,"In a separate incident, Kafr al-Sheikh Governo...",0,NaN,1618528739,NaN


After this, I want to check again the number of elements of the dataset.

In [83]:
# Summary of the cleaned dataset
rows, cols = df_egypt2011_clean.shape
print(f"Rows: {rows:,}")
print(f"Columns: {cols}")

print("\nColumn titles:")
print(df_egypt2011_clean.columns.tolist())

print("\nRow titles (index) preview:")
print(df_egypt2011_clean.index.tolist()[:20])

Rows: 2,323
Columns: 32

Column titles:
['event_id_cnty', 'event_date', 'year', 'time_precision', 'disorder_type', 'event_type', 'sub_event_type', 'actor1', 'assoc_actor_1', 'inter1', 'actor2', 'assoc_actor_2', 'inter2', 'interaction', 'civilian_targeting', 'iso', 'region', 'country', 'admin1', 'admin2', 'admin3', 'location', 'latitude', 'longitude', 'geo_precision', 'source', 'source_scale', 'notes', 'fatalities', 'tags', 'timestamp', 'population_best']

Row titles (index) preview:
[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19]


I want to check that all the event protests have latitude and longitude values, before conducting my analysis. In case there are missing values, I will exclude those rows.

In [84]:
# Check whether protest events have city/location and geolocation
protests = df_egypt2011_clean[df_egypt2011_clean["event_type"].astype(str).str.lower() == "protests"].copy()

total_protests = len(protests)
missing_location = protests["location"].isna() | (protests["location"].astype(str).str.strip() == "")
missing_lat = protests["latitude"].isna()
missing_lon = protests["longitude"].isna()
missing_geo = missing_lat | missing_lon

print(f"Total protest events: {total_protests:,}")
print(f"Missing city/location: {missing_location.sum():,}")
print(f"Missing latitude: {missing_lat.sum():,}")
print(f"Missing longitude: {missing_lon.sum():,}")
print(f"Missing geolocation (lat or lon): {missing_geo.sum():,}")

if missing_location.sum() == 0 and missing_geo.sum() == 0:
    print("All protest events have both location and geolocation.")
else:
    print("Not all protest events have complete location/geolocation data.")

Total protest events: 1,780
Missing city/location: 0
Missing latitude: 0
Missing longitude: 0
Missing geolocation (lat or lon): 0
All protest events have both location and geolocation.


All events protests have latitude and longitude values, therefore I will not have to exclude any rows.

Now I want to do the same data cleaning, but for my second dataset "ACLED_alsisi", which takes into account Egyptian protests from 2019 to 2020.

First, I want to see how many elements my second dataset has.

In [85]:
import re
from pathlib import Path

# Step 1: Load ACLED_alsisi dataset
alsisi_path = Path("..") / "raw_data" / "ACLED_alsisi.csv"
if not alsisi_path.exists():
    alsisi_path = Path("raw_data") / "ACLED_alsisi.csv"

df_alsisi = pd.read_csv(alsisi_path)
print(f"Loaded ACLED_alsisi: {len(df_alsisi):,} rows, {df_alsisi.shape[1]} columns")
df_alsisi.head()

Loaded ACLED_alsisi: 153 rows, 32 columns


,event_id_cnty,event_date,year,time_precision,disorder_type,event_type,sub_event_type,actor1,assoc_actor_1,inter1,...,latitude,longitude,geo_precision,source,source_scale,notes,fatalities,tags,timestamp,population_best
0,EGY10042,2019-09-22,2019,1,Political violence; Demonstrations,Protests,Excessive force against protesters,Protesters (Egypt),NaN,Protesters,...,29.9737,32.5263,1,AFP; EuroNews; Facebook,New media-International,"During the night of 22 Sep. 2019, security for...",0,crowd size=200,1618528470,84556
1,EGY10634,2019-12-20,2019,2,Demonstrations,Protests,Protest with intervention,Protesters (Egypt),Labor Group (Egypt),Protesters,...,30.2964,31.7463,1,FJ Portal,National,"On 20 December 2019, police intervened in a pr...",0,crowd size=no report,1618528543,16673
2,EGY10657,2019-12-15,2019,1,Demonstrations,Protests,Peaceful protest,Protesters (Egypt),NaN,Protesters,...,31.0392,30.4691,1,FJ Portal,National,"On 15 December 2019, protesters organizing und...",0,crowd size=no report,1618528545,122636
3,EGY10677,2020-01-21,2020,1,Demonstrations,Protests,Peaceful protest,Protesters (Egypt),Health Workers (Egypt),Protesters,...,28.1099,30.7503,1,Madamasr; El Fagr News,National,"On 21 January 2020, members of the General Ass...",0,crowd size=no report,1618528547,24137
4,EGY10678,2020-01-23,2020,1,Demonstrations,Protests,Peaceful protest,Protesters (Egypt),Journalists (Egypt),Protesters,...,30.0351,31.2140,1,El Fagr News; Masr Alarabia,National,"On 23 January 2020, journalists at Saba7 Newsp...",0,crowd size=no report,1764045636,25670


Once again, just like I did for my first dataset, I want to exclude the words 'israel', 'palestine' , 'palestinian', 'israeli'. I want to make sure that the protests in the dataset are domestic and against the government, rather than related to the bigger regional dynamics.  

I also remove mob violence.

In [86]:
# Step 2: Count whole-word mentions of exclusion keywords in full text
keywords_alsisi = ["israel", "israeli", "palestine", "palestinian"]
document_text_alsisi = " ".join(df_alsisi.fillna("").astype(str).to_numpy().ravel()).lower()
whole_word_counts_alsisi = {
    kw: len(re.findall(rf"\b{re.escape(kw)}\b", document_text_alsisi))
    for kw in keywords_alsisi
}

print("Whole-word occurrences in ACLED_alsisi:")
for kw, count in whole_word_counts_alsisi.items():
    print(f"{kw}: {count}")
print(f"Total whole-word matches: {sum(whole_word_counts_alsisi.values())}")

Whole-word occurrences in ACLED_alsisi:
israel: 2
israeli: 0
palestine: 2
palestinian: 1
Total whole-word matches: 5


In [87]:
# Step 3: Remove rows containing exclusion keywords
alsisi_exclude_pattern = r"\b(?:" + "|".join(map(re.escape, keywords_alsisi)) + r")\b"
row_text_alsisi = df_alsisi.fillna("").astype(str).agg(" ".join, axis=1)
mask_excluded_alsisi = row_text_alsisi.str.contains(alsisi_exclude_pattern, case=False, na=False, regex=True)

# Also remove mob violence at cleaning stage (before analysis/plots).
mask_mob_alsisi = (
    df_alsisi["sub_event_type"]
    .astype(str)
    .str.strip()
    .str.lower()
    .str.contains(r"\bmob violence\b", na=False, regex=True)
 )

mask_remove_alsisi = mask_excluded_alsisi | mask_mob_alsisi
df_alsisi_clean = df_alsisi.loc[~mask_remove_alsisi].copy()

removed_alsisi_keywords = mask_excluded_alsisi.sum()
removed_alsisi_mob_only = ((~mask_excluded_alsisi) & mask_mob_alsisi).sum()

print("Filtering summary:")
print(f"Original events: {len(df_alsisi):,}")
print(f"Removed events (keywords): {removed_alsisi_keywords:,}")
print(f"Removed events (mob violence): {removed_alsisi_mob_only:,}")
print(f"Removed events (total): {mask_remove_alsisi.sum():,}")
print(f"Remaining events: {len(df_alsisi_clean):,}")
df_alsisi_clean.head()

Filtering summary:
Original events: 153
Removed events (keywords): 2
Removed events (mob violence): 4
Removed events (total): 6
Remaining events: 147


,event_id_cnty,event_date,year,time_precision,disorder_type,event_type,sub_event_type,actor1,assoc_actor_1,inter1,...,latitude,longitude,geo_precision,source,source_scale,notes,fatalities,tags,timestamp,population_best
0,EGY10042,2019-09-22,2019,1,Political violence; Demonstrations,Protests,Excessive force against protesters,Protesters (Egypt),NaN,Protesters,...,29.9737,32.5263,1,AFP; EuroNews; Facebook,New media-International,"During the night of 22 Sep. 2019, security for...",0,crowd size=200,1618528470,84556
1,EGY10634,2019-12-20,2019,2,Demonstrations,Protests,Protest with intervention,Protesters (Egypt),Labor Group (Egypt),Protesters,...,30.2964,31.7463,1,FJ Portal,National,"On 20 December 2019, police intervened in a pr...",0,crowd size=no report,1618528543,16673
2,EGY10657,2019-12-15,2019,1,Demonstrations,Protests,Peaceful protest,Protesters (Egypt),NaN,Protesters,...,31.0392,30.4691,1,FJ Portal,National,"On 15 December 2019, protesters organizing und...",0,crowd size=no report,1618528545,122636
3,EGY10677,2020-01-21,2020,1,Demonstrations,Protests,Peaceful protest,Protesters (Egypt),Health Workers (Egypt),Protesters,...,28.1099,30.7503,1,Madamasr; El Fagr News,National,"On 21 January 2020, members of the General Ass...",0,crowd size=no report,1618528547,24137
4,EGY10678,2020-01-23,2020,1,Demonstrations,Protests,Peaceful protest,Protesters (Egypt),Journalists (Egypt),Protesters,...,30.0351,31.2140,1,El Fagr News; Masr Alarabia,National,"On 23 January 2020, journalists at Saba7 Newsp...",0,crowd size=no report,1764045636,25670


Now that I have deleted these events, I want to check again the elements in the dataset as well as ensure that the timeframe is correct.

In [88]:
# Step 4: Cleaned dataset shape
rows_alsisi, cols_alsisi = df_alsisi_clean.shape
print(f"Rows: {rows_alsisi:,}")
print(f"Columns: {cols_alsisi}")

print("\nColumn titles:")
print(df_alsisi_clean.columns.tolist())

Rows: 147
Columns: 32

Column titles:
['event_id_cnty', 'event_date', 'year', 'time_precision', 'disorder_type', 'event_type', 'sub_event_type', 'actor1', 'assoc_actor_1', 'inter1', 'actor2', 'assoc_actor_2', 'inter2', 'interaction', 'civilian_targeting', 'iso', 'region', 'country', 'admin1', 'admin2', 'admin3', 'location', 'latitude', 'longitude', 'geo_precision', 'source', 'source_scale', 'notes', 'fatalities', 'tags', 'timestamp', 'population_best']


In [89]:
# Step 6: Protest events time window
protests_dates_alsisi = pd.to_datetime(protests_alsisi["event_date"], errors="coerce")
start_date_alsisi = protests_dates_alsisi.min()
end_date_alsisi = protests_dates_alsisi.max()

print(f"Protest events time window: {start_date_alsisi.date()} to {end_date_alsisi.date()}")
print(f"Total span: {(end_date_alsisi - start_date_alsisi).days} days")

Protest events time window: 2019-09-20 to 2020-09-26
Total span: 372 days


I want to check that all protests have latitude and longitude values, so there is nothing missing when I will start doing my analysis.

In [90]:
# Step 5: Geolocation completeness for protest events
protests_alsisi = df_alsisi_clean[df_alsisi_clean["event_type"].astype(str).str.lower() == "protests"].copy()
missing_location_alsisi = protests_alsisi["location"].isna() | (protests_alsisi["location"].astype(str).str.strip() == "")
missing_lat_alsisi = protests_alsisi["latitude"].isna()
missing_lon_alsisi = protests_alsisi["longitude"].isna()
missing_geo_alsisi = missing_lat_alsisi | missing_lon_alsisi

print(f"Total protest events: {len(protests_alsisi):,}")
print(f"Missing city/location: {missing_location_alsisi.sum():,}")
print(f"Missing latitude: {missing_lat_alsisi.sum():,}")
print(f"Missing longitude: {missing_lon_alsisi.sum():,}")
print(f"Missing geolocation (lat or lon): {missing_geo_alsisi.sum():,}")

Total protest events: 137
Missing city/location: 0
Missing latitude: 0
Missing longitude: 0
Missing geolocation (lat or lon): 0


Now I can finally start with my data exploration and analysis!

In [91]:
import re
import pandas as pd
import plotly.express as px

# ACLED Egypt 2011: locations by subtype (protests + riots)
if "df_egypt2011_contentious" in globals():
    map_df = df_egypt2011_contentious.copy()
else:
    map_df = df_egypt2011_clean[
        df_egypt2011_clean["event_type"].astype(str).str.strip().str.lower().isin(["protests", "riots"])
    ].copy()

# Normalize subtype names to keep the four categories consistent.
def normalize_subtype(value):
    text = str(value).strip().lower()
    text = re.sub(r"\s+", " ", text)
    if "excessive" in text and "protester" in text:
        return "Excessive force against protesters"
    if "intervention" in text:
        return "Protest with intervention"
    if "violent" in text and ("demonstration" in text or "protest" in text):
        return "Violent demonstration"
    if "peaceful" in text and "protest" in text:
        return "Peaceful protest"
    return None

map_df["sub_event_type_norm"] = map_df["sub_event_type"].map(normalize_subtype)
map_df["latitude"] = pd.to_numeric(map_df["latitude"], errors="coerce")
map_df["longitude"] = pd.to_numeric(map_df["longitude"], errors="coerce")
map_df = map_df.dropna(subset=["latitude", "longitude", "sub_event_type_norm"])

# One dot per event using exact coordinates.
fig = px.scatter_map(
    map_df,
    lat="latitude",
    lon="longitude",
    color="sub_event_type_norm",
    hover_name="location" if "location" in map_df.columns else None,
    hover_data={"event_type": True, "event_date": True, "sub_event_type": True},
    color_discrete_map={
        "Protest with intervention": "#000000",
        "Excessive force against protesters": "#808080",
        "Peaceful protest": "#ff7f0e",
        "Violent demonstration": "#d62728",
    },
    category_orders={
        "sub_event_type_norm": [
            "Protest with intervention",
            "Excessive force against protesters",
            "Peaceful protest",
            "Violent demonstration",
        ]
    },
    title="ACLED Egypt 2011: Locations by subtype (protests + riots)",
    zoom=5,
    height=700,
)

fig.update_traces(marker=dict(size=7, opacity=0.7))
fig.update_layout(map_style="carto-positron", legend_title="Subtype")
fig.show()

print("Subtype counts shown on map:")
print(map_df["sub_event_type_norm"].value_counts())
print(f"Mapped events: {len(map_df):,}")

Subtype counts shown on map:
sub_event_type_norm
Peaceful protest                      1611
Violent demonstration                  543
Protest with intervention              114
Excessive force against protesters      55
Name: count, dtype: int64
Mapped events: 2,323


In [92]:
# Consistency check for the same analysis scope used in the map: protests + riots
contentious = df_egypt2011_clean[
    df_egypt2011_clean["event_type"].astype(str).str.strip().str.lower().isin(["protests", "riots"])
].copy()

print("Raw sub_event_type counts in df_egypt2011_clean (protests + riots):")
print(contentious["sub_event_type"].astype(str).str.strip().value_counts().head(15))

# Apply the same normalization used in the map cell.
import re

def normalize_subtype_debug(value):
    text = str(value).strip().lower()
    text = re.sub(r"\s+", " ", text)
    if "excessive" in text and "protester" in text:
        return "Excessive force against protesters"
    if "intervention" in text:
        return "Protest with intervention"
    if "violent" in text and ("demonstration" in text or "protest" in text):
        return "Violent demonstration"
    if "peaceful" in text and "protest" in text:
        return "Peaceful protest"
    return None

norm = contentious["sub_event_type"].map(normalize_subtype_debug)
print("\nNormalized map categories (protests + riots):")
print(norm.value_counts(dropna=False))

Raw sub_event_type counts in df_egypt2011_clean (protests + riots):
sub_event_type
Peaceful protest                      1611
Violent demonstration                  543
Protest with intervention              114
Excessive force against protesters      55
Name: count, dtype: int64

Normalized map categories (protests + riots):
sub_event_type
Peaceful protest                      1611
Violent demonstration                  543
Protest with intervention              114
Excessive force against protesters      55
Name: count, dtype: int64


In [93]:
# Final alignment check: raw vs cleaned, and the chosen analysis subset (protests + riots)
raw_all = df_egypt2011.copy()
clean_all = df_egypt2011_clean.copy()
clean_contentious = df_egypt2011_clean[
    df_egypt2011_clean["event_type"].astype(str).str.strip().str.lower().isin(["protests", "riots"])
].copy()

for frame_name, frame in [("RAW ALL", raw_all), ("CLEAN ALL", clean_all), ("CLEAN PROTESTS+RIOTS", clean_contentious)]:
    s = frame["sub_event_type"].astype(str).str.strip()
    event_type = frame["event_type"].astype(str).str.strip().str.lower()

    exact_violent_demo_all = s.str.lower().eq("violent demonstration").sum()
    exact_violent_demo_contentious = (
        s.str.lower().eq("violent demonstration") & event_type.isin(["protests", "riots"])
    ).sum()
    any_violent_all = s.str.lower().str.contains("violent", na=False).sum()
    any_violent_contentious = (
        s.str.lower().str.contains("violent", na=False) & event_type.isin(["protests", "riots"])
    ).sum()

    print(f"\n{frame_name}")
    print(f"Exact 'Violent demonstration' (all event types): {exact_violent_demo_all}")
    print(f"Exact 'Violent demonstration' (protests+riots): {exact_violent_demo_contentious}")
    print(f"Any sub_event_type containing 'violent' (all event types): {any_violent_all}")
    print(f"Any sub_event_type containing 'violent' (protests+riots): {any_violent_contentious}")

print("\nTop violent-like labels in RAW data:")
raw_labels = df_egypt2011["sub_event_type"].astype(str).str.strip()
print(raw_labels[raw_labels.str.lower().str.contains("violent", na=False)].value_counts().head(20))


RAW ALL
Exact 'Violent demonstration' (all event types): 549
Exact 'Violent demonstration' (protests+riots): 549
Any sub_event_type containing 'violent' (all event types): 549
Any sub_event_type containing 'violent' (protests+riots): 549

CLEAN ALL
Exact 'Violent demonstration' (all event types): 543
Exact 'Violent demonstration' (protests+riots): 543
Any sub_event_type containing 'violent' (all event types): 543
Any sub_event_type containing 'violent' (protests+riots): 543

CLEAN PROTESTS+RIOTS
Exact 'Violent demonstration' (all event types): 543
Exact 'Violent demonstration' (protests+riots): 543
Any sub_event_type containing 'violent' (all event types): 543
Any sub_event_type containing 'violent' (protests+riots): 543

Top violent-like labels in RAW data:
sub_event_type
Violent demonstration    549
Name: count, dtype: int64


In [94]:
# Reusable analysis subset for maps/charts: protests + riots
# This keeps the cleaning pipeline intact and standardizes downstream analysis scope.
df_egypt2011_contentious = df_egypt2011_clean[
    df_egypt2011_clean["event_type"].astype(str).str.strip().str.lower().isin(["protests", "riots"])
].copy()

print("Event type counts in analysis subset:")
print(df_egypt2011_contentious["event_type"].astype(str).str.strip().value_counts())
print(f"Rows in protests+riots subset: {len(df_egypt2011_contentious):,}")

Event type counts in analysis subset:
event_type
Protests    1780
Riots        543
Name: count, dtype: int64
Rows in protests+riots subset: 2,323


Now I will plot the same subtype location map for the ACLED_alsisi dataset (using the cleaned data and the same color/style settings).

In [97]:
import re
import pandas as pd
import plotly.express as px

# ACLED alsisi: locations by subtype (protests + riots)
df_alsisi_contentious = df_alsisi_clean[
    df_alsisi_clean["event_type"].astype(str).str.strip().str.lower().isin(["protests", "riots"])
].copy()

def normalize_subtype_alsisi(value):
    text = str(value).strip().lower()
    text = re.sub(r"\s+", " ", text)
    if "excessive" in text and "protester" in text:
        return "Excessive force against protesters"
    if "intervention" in text:
        return "Protest with intervention"
    if "violent" in text and ("demonstration" in text or "protest" in text):
        return "Violent demonstration"
    if "peaceful" in text and "protest" in text:
        return "Peaceful protest"
    return None

map_df_alsisi = df_alsisi_contentious.copy()
map_df_alsisi["sub_event_type_norm"] = map_df_alsisi["sub_event_type"].map(normalize_subtype_alsisi)
map_df_alsisi["latitude"] = pd.to_numeric(map_df_alsisi["latitude"], errors="coerce")
map_df_alsisi["longitude"] = pd.to_numeric(map_df_alsisi["longitude"], errors="coerce")
map_df_alsisi = map_df_alsisi.dropna(subset=["latitude", "longitude", "sub_event_type_norm"])

# One dot per event using exact coordinates.
fig_alsisi = px.scatter_map(
    map_df_alsisi,
    lat="latitude",
    lon="longitude",
    color="sub_event_type_norm",
    hover_name="location" if "location" in map_df_alsisi.columns else None,
    hover_data={"event_type": True, "event_date": True, "sub_event_type": True},
    color_discrete_map={
        "Protest with intervention": "#000000",
        "Excessive force against protesters": "#808080",
        "Peaceful protest": "#ff7f0e",
        "Violent demonstration": "#d62728",
    },
    category_orders={
        "sub_event_type_norm": [
            "Protest with intervention",
            "Excessive force against protesters",
            "Peaceful protest",
            "Violent demonstration",
        ]
    },
    title="ACLED alsisi (2019-2020): Locations by subtype (protests + riots)",
    zoom=5,
    height=700,
)

fig_alsisi.update_traces(marker=dict(size=7, opacity=0.7))
fig_alsisi.update_layout(map_style="carto-positron", legend_title="Subtype")
fig_alsisi.show()

print("Subtype counts shown on map (alsisi):")
print(map_df_alsisi["sub_event_type_norm"].value_counts())
print(f"Mapped events: {len(map_df_alsisi):,}")

Subtype counts shown on map (alsisi):
sub_event_type_norm
Peaceful protest                      77
Protest with intervention             55
Violent demonstration                 10
Excessive force against protesters     5
Name: count, dtype: int64
Mapped events: 147


Interactive monthly maps (time slider) for both datasets, using the four protest-related sub-event categories. Dot size represents how many events happened at the same location in that month.

In [121]:
import re
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

def normalize_subtype_4(value):
    text = str(value).strip().lower()
    text = re.sub(r"\s+", " ", text)
    if "excessive" in text and "protester" in text:
        return "Excessive force against protesters"
    if "intervention" in text:
        return "Protest with intervention"
    if "violent" in text and ("demonstration" in text or "protest" in text):
        return "Violent demonstration"
    if "peaceful" in text and "protest" in text:
        return "Peaceful protest"
    return None

def build_monthly_slider_map(df_clean, title):
    allowed_subtypes = [
        "Peaceful protest",
        "Protest with intervention",
        "Violent demonstration",
        "Excessive force against protesters",
    ]

    color_map = {
        "Protest with intervention": "#000000",
        "Excessive force against protesters": "#808080",
        "Peaceful protest": "#ff7f0e",
        "Violent demonstration": "#d62728",
    }

    base = df_clean[
        df_clean["event_type"].astype(str).str.strip().str.lower().isin(["protests", "riots"])
    ].copy()

    base["sub_event_type_norm"] = base["sub_event_type"].map(normalize_subtype_4)
    base["latitude"] = pd.to_numeric(base["latitude"], errors="coerce")
    base["longitude"] = pd.to_numeric(base["longitude"], errors="coerce")
    base["event_date"] = pd.to_datetime(base["event_date"], errors="coerce")
    base = base.dropna(subset=["event_date", "latitude", "longitude", "sub_event_type_norm"])
    base = base[base["sub_event_type_norm"].isin(allowed_subtypes)].copy()
    base["month"] = base["event_date"].dt.to_period("M").astype(str)

    monthly_points = (
        base.groupby(["month", "latitude", "longitude", "location", "sub_event_type_norm"], dropna=False)
        .size()
        .reset_index(name="events_at_location_month")
    )

    monthly_points["size_visual"] = monthly_points["events_at_location_month"].pow(1.6)

    fig = px.scatter_map(
        monthly_points,
        lat="latitude",
        lon="longitude",
        color="sub_event_type_norm",
        size="size_visual",
        size_max=46,
        animation_frame="month",
        hover_name="location",
        hover_data={
            "events_at_location_month": True,
            "sub_event_type_norm": True,
            "month": True,
            "size_visual": False,
            "latitude": False,
            "longitude": False,
        },
        color_discrete_map=color_map,
        category_orders={"sub_event_type_norm": allowed_subtypes},
        title=title,
        zoom=5,
        height=720,
    )

    # Keep the legend restricted to the four subtypes.
    base_trace_count = len(fig.data)
    max_size = monthly_points["size_visual"].max() if len(monthly_points) else 1
    sizeref = (2.0 * max_size) / (46.0 ** 2)
    for i in range(base_trace_count):
        fig.data[i].showlegend = False
        fig.data[i].marker.opacity = 0.88
        fig.data[i].marker.sizemode = "area"
        fig.data[i].marker.sizemin = 7
        fig.data[i].marker.sizeref = sizeref

    for subtype in allowed_subtypes:
        fig.add_trace(
            go.Scattermap(
                lat=[None],
                lon=[None],
                mode="markers",
                marker=dict(size=12, color=color_map[subtype]),
                name=subtype,
                visible="legendonly",
                hoverinfo="skip",
                showlegend=True,
            )
        )

    fig.update_layout(
        map_style="carto-positron",
        legend_title="Sub-event type",
        legend=dict(itemclick=False, itemdoubleclick=False),
    )

    fig.show()

    print(f"\n{title} -> subtype counts:")
    print(base["sub_event_type_norm"].value_counts())
    print(f"{title} -> total mapped events: {len(base):,}")
    print(f"{title} -> monthly location bubbles: {len(monthly_points):,}")

# Graph 1: 2011 dataset
build_monthly_slider_map(
    df_egypt2011_clean,
    "ACLED Egypt 2011: Monthly interactive map (4 sub-event types, size = events)",
)

# Graph 2: alsisi dataset
build_monthly_slider_map(
    df_alsisi_clean,
    "ACLED alsisi 2019-2020: Monthly interactive map (4 sub-event types, size = events)",
)


ACLED Egypt 2011: Monthly interactive map (4 sub-event types, size = events) -> subtype counts:
sub_event_type_norm
Peaceful protest                      1611
Violent demonstration                  543
Protest with intervention              114
Excessive force against protesters      55
Name: count, dtype: int64
ACLED Egypt 2011: Monthly interactive map (4 sub-event types, size = events) -> total mapped events: 2,323
ACLED Egypt 2011: Monthly interactive map (4 sub-event types, size = events) -> monthly location bubbles: 1,097



ACLED alsisi 2019-2020: Monthly interactive map (4 sub-event types, size = events) -> subtype counts:
sub_event_type_norm
Peaceful protest                      77
Protest with intervention             55
Violent demonstration                 10
Excessive force against protesters     5
Name: count, dtype: int64
ACLED alsisi 2019-2020: Monthly interactive map (4 sub-event types, size = events) -> total mapped events: 147
ACLED alsisi 2019-2020: Monthly interactive map (4 sub-event types, size = events) -> monthly location bubbles: 107


In [104]:
# Quick debug: when does 'Excessive force against protesters' appear in Egypt 2011 map scope?
def normalize_subtype_4_debug(value):
    text = str(value).strip().lower()
    text = re.sub(r"\s+", " ", text)
    if "excessive" in text and "protester" in text:
        return "Excessive force against protesters"
    if "intervention" in text:
        return "Protest with intervention"
    if "violent" in text and ("demonstration" in text or "protest" in text):
        return "Violent demonstration"
    if "peaceful" in text and "protest" in text:
        return "Peaceful protest"
    return None

dbg = df_egypt2011_clean[
    df_egypt2011_clean["event_type"].astype(str).str.strip().str.lower().isin(["protests", "riots"])
].copy()

dbg["event_date"] = pd.to_datetime(dbg["event_date"], errors="coerce")
dbg["sub_event_type_norm"] = dbg["sub_event_type"].map(normalize_subtype_4_debug)
excessive = dbg[dbg["sub_event_type_norm"].eq("Excessive force against protesters")].copy()
excessive["month"] = excessive["event_date"].dt.to_period("M").astype(str)

print(f"Total excessive-force events in Egypt 2011 scope: {len(excessive)}")
print("Months containing excessive-force events:")
print(excessive["month"].value_counts().sort_index())

Total excessive-force events in Egypt 2011 scope: 55
Months containing excessive-force events:
month
2011-02    10
2011-03     3
2011-04     2
2011-07     5
2011-08     2
2011-10     2
2011-11     3
2011-12     1
2012-05     1
2012-06     1
2012-11     1
2012-12     2
2013-01     2
2013-02     5
2013-06     4
2013-07     9
2013-08     2
Name: count, dtype: int64


In [123]:
import re
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

def normalize_subtype_4(value):
    text = str(value).strip().lower()
    text = re.sub(r"\s+", " ", text)
    if "excessive" in text and "protester" in text:
        return "Excessive force against protesters"
    if "intervention" in text:
        return "Protest with intervention"
    if "violent" in text and ("demonstration" in text or "protest" in text):
        return "Violent demonstration"
    if "peaceful" in text and "protest" in text:
        return "Peaceful protest"
    return None

def build_monthly_slider_map(df_clean, title):
    allowed_subtypes = [
        "Peaceful protest",
        "Protest with intervention",
        "Violent demonstration",
        "Excessive force against protesters",
    ]

    color_map = {
        "Protest with intervention": "#000000",
        "Excessive force against protesters": "#808080",
        "Peaceful protest": "#ff7f0e",
        "Violent demonstration": "#d62728",
    }

    base = df_clean[
        df_clean["event_type"].astype(str).str.strip().str.lower().isin(["protests", "riots"])
    ].copy()

    base["sub_event_type_norm"] = base["sub_event_type"].map(normalize_subtype_4)
    base["latitude"] = pd.to_numeric(base["latitude"], errors="coerce")
    base["longitude"] = pd.to_numeric(base["longitude"], errors="coerce")
    base["event_date"] = pd.to_datetime(base["event_date"], errors="coerce")
    base = base.dropna(subset=["event_date", "latitude", "longitude", "sub_event_type_norm"])
    base = base[base["sub_event_type_norm"].isin(allowed_subtypes)].copy()
    base["month"] = base["event_date"].dt.to_period("M").astype(str)

    monthly_points = (
        base.groupby(["month", "latitude", "longitude", "location", "sub_event_type_norm"], dropna=False)
        .size()
        .reset_index(name="events_at_location_month")
    )

    monthly_points["size_visual"] = monthly_points["events_at_location_month"].pow(1.6)

    fig = px.scatter_map(
        monthly_points,
        lat="latitude",
        lon="longitude",
        color="sub_event_type_norm",
        size="size_visual",
        size_max=46,
        animation_frame="month",
        hover_name="location",
        hover_data={
            "events_at_location_month": True,
            "sub_event_type_norm": True,
            "month": True,
            "size_visual": False,
            "latitude": False,
            "longitude": False,
        },
        color_discrete_map=color_map,
        category_orders={"sub_event_type_norm": allowed_subtypes},
        title=title,
        zoom=5,
        height=720,
    )

    # CRITICAL FIX: Remove all traces and rebuild manually
    # This prevents animation frames from auto-generating legend entries
    fig.data = []
    
    # Get unique months for animation frames
    months = sorted(monthly_points["month"].unique())
    
    # Build traces for the first month
    first_month = months[0]
    first_month_data = monthly_points[monthly_points["month"] == first_month]
    
    # Create one trace per subtype for proper legend control
    for subtype in allowed_subtypes:
        subtype_data = first_month_data[first_month_data["sub_event_type_norm"] == subtype]
        if len(subtype_data) > 0:
            fig.add_trace(
                go.Scattermap(
                    lat=subtype_data["latitude"],
                    lon=subtype_data["longitude"],
                    mode="markers",
                    marker=dict(
                        size=subtype_data["size_visual"],
                        color=color_map[subtype],
                        sizemode="area",
                        sizemin=6,
                        opacity=0.85,
                    ),
                    name=subtype,
                    hovertext=subtype_data.apply(
                        lambda row: f"Location: {row['location']}<br>Events: {row['events_at_location_month']}<br>Type: {row['sub_event_type_norm']}<br>Month: {row['month']}",
                        axis=1
                    ),
                    hoverinfo="text",
                    showlegend=True,
                )
            )
        else:
            # Add empty trace to keep legend entry
            fig.add_trace(
                go.Scattermap(
                    lat=[None],
                    lon=[None],
                    mode="markers",
                    marker=dict(
                        size=12,
                        color=color_map[subtype],
                    ),
                    name=subtype,
                    showlegend=True,
                )
            )
    
    # Build animation frames
    frames = []
    for month in months[1:]:  # Skip first month (already shown)
        month_data = monthly_points[monthly_points["month"] == month]
        frame_traces = []
        
        for subtype in allowed_subtypes:
            subtype_data = month_data[month_data["sub_event_type_norm"] == subtype]
            if len(subtype_data) > 0:
                frame_traces.append(
                    go.Scattermap(
                        lat=subtype_data["latitude"],
                        lon=subtype_data["longitude"],
                        mode="markers",
                        marker=dict(
                            size=subtype_data["size_visual"],
                            color=color_map[subtype],
                            sizemode="area",
                            sizemin=6,
                            opacity=0.85,
                        ),
                        hovertext=subtype_data.apply(
                            lambda row: f"Location: {row['location']}<br>Events: {row['events_at_location_month']}<br>Type: {row['sub_event_type_norm']}<br>Month: {row['month']}",
                            axis=1
                        ),
                        hoverinfo="text",
                        showlegend=False,
                    )
                )
            else:
                frame_traces.append(
                    go.Scattermap(
                        lat=[None],
                        lon=[None],
                        mode="markers",
                        marker=dict(
                            size=12,
                            color=color_map[subtype],
                        ),
                        showlegend=False,
                    )
                )
        
        frames.append(go.Frame(data=frame_traces, name=month))
    
    fig.frames = frames
    
    # Update layout with animation controls
    fig.update_layout(
        map_style="carto-positron",
        legend_title="Sub-event type",
        legend=dict(itemclick=False, itemdoubleclick=False),
        updatemenus=[{
            "type": "buttons",
            "showactive": False,
            "buttons": [
                {
                    "label": "Play",
                    "method": "animate",
                    "args": [None, {
                        "frame": {"duration": 800, "redraw": True},
                        "fromcurrent": True,
                        "transition": {"duration": 400}
                    }]
                },
                {
                    "label": "Pause",
                    "method": "animate",
                    "args": [[None], {
                        "frame": {"duration": 0, "redraw": False},
                        "mode": "immediate",
                        "transition": {"duration": 0}
                    }]
                }
            ]
        }],
        sliders=[{
            "active": 0,
            "steps": [{
                "label": month,
                "method": "animate",
                "args": [[month], {
                    "frame": {"duration": 0, "redraw": True},
                    "mode": "immediate",
                    "transition": {"duration": 0}
                }]
            } for month in months],
            "x": 0.1,
            "len": 0.9,
        }]
    )
    
    fig.show()

    print(f"\n{title} -> subtype counts:")
    print(base["sub_event_type_norm"].value_counts())
    print(f"{title} -> total mapped events: {len(base):,}")
    print(f"{title} -> monthly location bubbles: {len(monthly_points):,}")

# Graph 1: 2011 dataset
build_monthly_slider_map(
    df_egypt2011_clean,
    "ACLED Egypt 2011: Monthly interactive map (4 sub-event types, size = events)",
)

# Graph 2: alsisi dataset
build_monthly_slider_map(
    df_alsisi_clean,
    "ACLED alsisi 2019-2020: Monthly interactive map (4 sub-event types, size = events)",

_IncompleteInputError: incomplete input (2056505876.py, line 238)

In [112]:
import re
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

def normalize_subtype_4(value):
    text = str(value).strip().lower()
    text = re.sub(r"\s+", " ", text)
    if "excessive" in text and "protester" in text:
        return "Excessive force against protesters"
    if "intervention" in text:
        return "Protest with intervention"
    if "violent" in text and ("demonstration" in text or "protest" in text):
        return "Violent demonstration"
    if "peaceful" in text and "protest" in text:
        return "Peaceful protest"
    return None

def build_monthly_slider_map(df_clean, title):
    allowed_subtypes = [
        "Peaceful protest",
        "Protest with intervention",
        "Violent demonstration",
        "Excessive force against protesters",
    ]

    color_map = {
        "Protest with intervention": "#000000",
        "Excessive force against protesters": "#808080",
        "Peaceful protest": "#ff7f0e",
        "Violent demonstration": "#d62728",
    }

    base = df_clean[
        df_clean["event_type"].astype(str).str.strip().str.lower().isin(["protests", "riots"])
    ].copy()

    base["sub_event_type_norm"] = base["sub_event_type"].map(normalize_subtype_4)
    base["latitude"] = pd.to_numeric(base["latitude"], errors="coerce")
    base["longitude"] = pd.to_numeric(base["longitude"], errors="coerce")
    base["event_date"] = pd.to_datetime(base["event_date"], errors="coerce")
    base = base.dropna(subset=["event_date", "latitude", "longitude", "sub_event_type_norm"])
    base = base[base["sub_event_type_norm"].isin(allowed_subtypes)].copy()
    base["month"] = base["event_date"].dt.to_period("M").astype(str)

    monthly_points = (
        base.groupby(["month", "latitude", "longitude", "location", "sub_event_type_norm"], dropna=False)
        .size()
        .reset_index(name="events_at_location_month")
    )

    monthly_points["size_visual"] = monthly_points["events_at_location_month"].pow(1.6)

    fig = px.scatter_map(
        monthly_points,
        lat="latitude",
        lon="longitude",
        color="sub_event_type_norm",
        size="size_visual",
        size_max=46,
        animation_frame="month",
        hover_name="location",
        hover_data={
            "events_at_location_month": True,
            "sub_event_type_norm": True,
            "month": True,
            "size_visual": False,
            "latitude": False,
            "longitude": False,
        },
        color_discrete_map=color_map,
        category_orders={"sub_event_type_norm": allowed_subtypes},
        title=title,
        zoom=5,
        height=720,
    )

    # CRITICAL FIX: Remove all traces and rebuild manually
    # This prevents animation frames from auto-generating legend entries
    fig.data = []
    
    # Get unique months for animation frames
    months = sorted(monthly_points["month"].unique())
    
    # Build traces for the first month
    first_month = months[0]
    first_month_data = monthly_points[monthly_points["month"] == first_month]
    
    # Create one trace per subtype for proper legend control
    for subtype in allowed_subtypes:
        subtype_data = first_month_data[first_month_data["sub_event_type_norm"] == subtype]
        if len(subtype_data) > 0:
            fig.add_trace(
                go.Scattermap(
                    lat=subtype_data["latitude"],
                    lon=subtype_data["longitude"],
                    mode="markers",
                    marker=dict(
                        size=subtype_data["size_visual"],
                        color=color_map[subtype],
                        sizemode="area",
                        sizemin=6,
                        opacity=0.85,
                    ),
                    name=subtype,
                    hovertext=subtype_data.apply(
                        lambda row: f"Location: {row['location']}<br>Events: {row['events_at_location_month']}<br>Type: {row['sub_event_type_norm']}<br>Month: {row['month']}",
                        axis=1
                    ),
                    hoverinfo="text",
                    showlegend=True,
                )
            )
        else:
            # Add empty trace to keep legend entry
            fig.add_trace(
                go.Scattermap(
                    lat=[None],
                    lon=[None],
                    mode="markers",
                    marker=dict(
                        size=12,
                        color=color_map[subtype],
                    ),
                    name=subtype,
                    showlegend=True,
                )
            )
    
    # Build animation frames
    frames = []
    for month in months[1:]:  # Skip first month (already shown)
        month_data = monthly_points[monthly_points["month"] == month]
        frame_traces = []
        
        for subtype in allowed_subtypes:
            subtype_data = month_data[month_data["sub_event_type_norm"] == subtype]
            if len(subtype_data) > 0:
                frame_traces.append(
                    go.Scattermap(
                        lat=subtype_data["latitude"],
                        lon=subtype_data["longitude"],
                        mode="markers",
                        marker=dict(
                            size=subtype_data["size_visual"],
                            color=color_map[subtype],
                            sizemode="area",
                            sizemin=6,
                            opacity=0.85,
                        ),
                        hovertext=subtype_data.apply(
                            lambda row: f"Location: {row['location']}<br>Events: {row['events_at_location_month']}<br>Type: {row['sub_event_type_norm']}<br>Month: {row['month']}",
                            axis=1
                        ),
                        hoverinfo="text",
                        showlegend=False,
                    )
                )
            else:
                frame_traces.append(
                    go.Scattermap(
                        lat=[None],
                        lon=[None],
                        mode="markers",
                        marker=dict(
                            size=12,
                            color=color_map[subtype],
                        ),
                        showlegend=False,
                    )
                )
        
        frames.append(go.Frame(data=frame_traces, name=month))
    
    fig.frames = frames
    
    # Update layout with animation controls
    fig.update_layout(
        map_style="carto-positron",
        legend_title="Sub-event type",
        legend=dict(itemclick=False, itemdoubleclick=False),
        updatemenus=[{
            "type": "buttons",
            "showactive": False,
            "buttons": [
                {
                    "label": "Play",
                    "method": "animate",
                    "args": [None, {
                        "frame": {"duration": 800, "redraw": True},
                        "fromcurrent": True,
                        "transition": {"duration": 400}
                    }]
                },
                {
                    "label": "Pause",
                    "method": "animate",
                    "args": [[None], {
                        "frame": {"duration": 0, "redraw": False},
                        "mode": "immediate",
                        "transition": {"duration": 0}
                    }]
                }
            ]
        }],
        sliders=[{
            "active": 0,
            "steps": [{
                "label": month,
                "method": "animate",
                "args": [[month], {
                    "frame": {"duration": 0, "redraw": True},
                    "mode": "immediate",
                    "transition": {"duration": 0}
                }]
            } for month in months],
            "x": 0.1,
            "len": 0.9,
        }]
    )
    
    fig.show()

    print(f"\n{title} -> subtype counts:")
    print(base["sub_event_type_norm"].value_counts())
    print(f"{title} -> total mapped events: {len(base):,}")
    print(f"{title} -> monthly location bubbles: {len(monthly_points):,}")

# Graph 1: 2011 dataset
build_monthly_slider_map(
    df_egypt2011_clean,
    "ACLED Egypt 2011: Monthly interactive map (4 sub-event types, size = events)",
)

# Graph 2: alsisi dataset
build_monthly_slider_map(
    df_alsisi_clean,
    "ACLED alsisi 2019-2020: Monthly interactive map (4 sub-event types, size = events)",
)


ACLED Egypt 2011: Monthly interactive map (4 sub-event types, size = events) -> subtype counts:
sub_event_type_norm
Peaceful protest                      1611
Violent demonstration                  543
Protest with intervention              114
Excessive force against protesters      55
Name: count, dtype: int64
ACLED Egypt 2011: Monthly interactive map (4 sub-event types, size = events) -> total mapped events: 2,323
ACLED Egypt 2011: Monthly interactive map (4 sub-event types, size = events) -> monthly location bubbles: 1,097



ACLED alsisi 2019-2020: Monthly interactive map (4 sub-event types, size = events) -> subtype counts:
sub_event_type_norm
Peaceful protest                      77
Protest with intervention             55
Violent demonstration                 10
Excessive force against protesters     5
Name: count, dtype: int64
ACLED alsisi 2019-2020: Monthly interactive map (4 sub-event types, size = events) -> total mapped events: 147
ACLED alsisi 2019-2020: Monthly interactive map (4 sub-event types, size = events) -> monthly location bubbles: 107


In [116]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

class EgyptProtestsComparison:
    """
    Comprehensive comparison of Egypt protests: 2011 vs 2019
    """
    
    def __init__(self, df_2011, df_2019):
        self.df_2011 = df_2011.copy()
        self.df_2019 = df_2019.copy()
        
        # Preprocess both datasets
        for df in [self.df_2011, self.df_2019]:
            df['event_date'] = pd.to_datetime(df['event_date'], errors='coerce')
            df['year'] = df['event_date'].dt.year
            df['month'] = df['event_date'].dt.month
            df['day_of_week'] = df['event_date'].dt.day_name()
            df['week'] = df['event_date'].dt.isocalendar().week
            
            # Categorize violence
            df['violence_level'] = df['sub_event_type'].apply(self._categorize_violence)
            
            # Add period label
            df['period'] = '2011' if df is self.df_2011 else '2019'
        
        # Combine for some analyses
        self.df_combined = pd.concat([self.df_2011, self.df_2019], ignore_index=True)
    
    def _categorize_violence(self, sub_event):
        sub_event = str(sub_event).lower().strip()
        
        if any(term in sub_event for term in ['excessive force', 'violent', 'armed', 'attack', 
                                               'lethal', 'deadly', 'explosive', 'shelling', 'bomb']):
            return 'High'
        elif any(term in sub_event for term in ['intervention', 'dispersal', 'clash', 'riot',
                                                'tear gas', 'rubber', 'stone']):
            return 'Medium'
        elif any(term in sub_event for term in ['peaceful', 'silent', 'march', 'rally', 'strike']):
            return 'Low/Peaceful'
        else:
            return 'Other'
    
    # ===== 1. TEMPORAL PATTERNS =====
    
    def analyze_temporal_patterns(self):
        """Compare timing, seasonality, and intensity over time"""
        
        print("\n" + "="*80)
        print("TEMPORAL PATTERN ANALYSIS")
        print("="*80)
        
        # 1.1 Monthly comparison of event counts
        monthly_2011 = self.df_2011.groupby('month').size()
        monthly_2019 = self.df_2019.groupby('month').size()
        
        # 1.2 Peak protest days analysis
        print("\n1. Peak Protest Days:")
        for period, df in [('2011', self.df_2011), ('2019', self.df_2019)]:
            daily_counts = df.groupby('event_date').size()
            print(f"\n{period} - Top 5 Most Active Days:")
            print(daily_counts.nlargest(5))
        
        # 1.3 Day of week patterns
        day_patterns = pd.crosstab(
            self.df_combined['day_of_week'], 
            self.df_combined['period'],
            normalize='columns'
        ) * 100
        
        print("\n2. Day of Week Distribution (%):")
        print(day_patterns.round(2))
        
        # 1.4 Weekend vs Weekday violence
        self.df_combined['is_weekend'] = self.df_combined['day_of_week'].isin(['Friday', 'Saturday'])
        weekend_violence = pd.crosstab(
            self.df_combined['is_weekend'], 
            [self.df_combined['period'], self.df_combined['violence_level']],
            normalize='columns'
        ) * 100
        
        print("\n3. Weekend vs Weekday Violence (%):")
        print(weekend_violence.round(2))
        
        # Visualize temporal patterns
        fig = make_subplots(
            rows=3, cols=2,
            subplot_titles=(
                'Monthly Event Distribution',
                'Cumulative Events Over Time',
                'Violence Intensity Timeline',
                'Day of Week Patterns',
                'Protest Duration (if available)',
                'Event Clustering (events per week)'
            )
        )
        
        # Monthly distribution
        fig.add_trace(
            go.Scatter(x=monthly_2011.index, y=monthly_2011.values, 
                      mode='lines+markers', name='2011', line=dict(color='#d62728')),
            row=1, col=1
        )
        fig.add_trace(
            go.Scatter(x=monthly_2019.index, y=monthly_2019.values,
                      mode='lines+markers', name='2019', line=dict(color='#1f77b4')),
            row=1, col=1
        )
        
        # Cumulative events
        for period, df, color in [('2011', self.df_2011, '#d62728'), ('2019', self.df_2019, '#1f77b4')]:
            daily = df.groupby('event_date').size().sort_index().cumsum()
            fig.add_trace(
                go.Scatter(x=daily.index, y=daily.values, mode='lines',
                          name=f'{period} cumulative', line=dict(color=color, width=2)),
                row=1, col=2
            )
        
        # Violence timeline
        for period, df, color in [('2011', self.df_2011, '#d62728'), ('2019', self.df_2019, '#1f77b4')]:
            daily_violence = df.groupby('event_date')['violence_level'].apply(
                lambda x: (x == 'High').sum() / len(x) * 100 if len(x) > 0 else 0
            )
            fig.add_trace(
                go.Scatter(x=daily_violence.index, y=daily_violence.values,
                          mode='lines', name=f'{period} high violence %',
                          line=dict(color=color, dash='dot')),
                row=2, col=1
            )
        
        # Day of week
        for period, color in [('2011', '#d62728'), ('2019', '#1f77b4')]:
            dow = day_patterns[period]
            fig.add_trace(
                go.Bar(x=dow.index, y=dow.values, name=period,
                      marker_color=color, opacity=0.7),
                row=2, col=2
            )
        
        # Weekly clustering
        for period, df, color in [('2011', self.df_2011, '#d62728'), ('2019', self.df_2019, '#1f77b4')]:
            weekly = df.groupby(['year', 'week']).size()
            week_numbers = list(range(1, len(weekly) + 1))
            fig.add_trace(
                go.Bar(x=week_numbers, y=weekly.values, name=f'{period} weekly',
                      marker_color=color, opacity=0.7),
                row=3, col=1
            )
        
        fig.update_layout(height=1000, title_text="Temporal Pattern Analysis: 2011 vs 2019")
        fig.show()
    
    # ===== 2. GEOGRAPHIC PATTERNS =====
    
    def analyze_geographic_patterns(self):
        """Compare spatial distribution, hotspots, and diffusion"""
        
        print("\n" + "="*80)
        print("GEOGRAPHIC PATTERN ANALYSIS")
        print("="*80)
        
        # 2.1 Top locations
        for period, df in [('2011', self.df_2011), ('2019', self.df_2019)]:
            print(f"\n{period} - Top 10 Protest Locations:")
            print(df['location'].value_counts().head(10))
        
        # 2.2 Geographic concentration (Herfindahl index)
        def location_concentration(df):
            loc_counts = df['location'].value_counts()
            total = loc_counts.sum()
            hhi = ((loc_counts / total) ** 2).sum()
            return hhi
        
        hhi_2011 = location_concentration(self.df_2011)
        hhi_2019 = location_concentration(self.df_2019)
        
        print(f"\nGeographic Concentration (HHI):")
        print(f"2011: {hhi_2011:.4f}")
        print(f"2019: {hhi_2019:.4f}")
        print(f"Higher HHI = more concentrated in fewer locations")
        
        # 2.3 Violence by governorate/region
        if 'admin1' in self.df_combined.columns:
            region_violence = pd.crosstab(
                self.df_combined['admin1'],
                [self.df_combined['period'], self.df_combined['violence_level']],
                normalize='index'
            ) * 100
            
            print("\nViolence by Region (%):")
            print(region_violence.round(2))
        
        # 2.4 New locations in 2019
        locations_2011 = set(self.df_2011['location'].unique())
        locations_2019 = set(self.df_2019['location'].unique())
        
        new_locations = locations_2019 - locations_2011
        disappeared_locations = locations_2011 - locations_2019
        
        print(f"\nProtest Location Changes:")
        print(f"Unique locations 2011: {len(locations_2011)}")
        print(f"Unique locations 2019: {len(locations_2019)}")
        print(f"New locations in 2019: {len(new_locations)}")
        print(f"Locations that disappeared: {len(disappeared_locations)}")
        
        if len(new_locations) > 0:
            print(f"\nSample of new locations: {list(new_locations)[:10]}")
        
        # Visualize geography
        fig = make_subplots(
            rows=2, cols=2,
            subplot_titles=(
                'Top 10 Locations Comparison',
                'Location Distribution (by event count)',
                'Violence by Region (if available)',
                'Spatial Diffusion Pattern'
            )
        )
        
        # Top locations
        locs_2011 = self.df_2011['location'].value_counts().head(10)
        locs_2019 = self.df_2019['location'].value_counts().head(10)
        
        fig.add_trace(
            go.Bar(y=locs_2011.index, x=locs_2011.values, name='2011',
                  orientation='h', marker_color='#d62728'),
            row=1, col=1
        )
        fig.add_trace(
            go.Bar(y=locs_2019.index, x=locs_2019.values, name='2019',
                  orientation='h', marker_color='#1f77b4'),
            row=1, col=1
        )
        
        # Location distribution
        for period, df, color in [('2011', self.df_2011, '#d62728'), ('2019', self.df_2019, '#1f77b4')]:
            loc_dist = df['location'].value_counts().values
            fig.add_trace(
                go.Histogram(x=loc_dist, name=f'{period} cities by event count',
                            marker_color=color, opacity=0.7, nbinsx=20),
                row=1, col=2
            )
        
        # Region violence (if available)
        if 'admin1' in self.df_combined.columns:
            regions = region_violence.index[:15]  # Top 15 regions
            for period, color in [('2011', '#d62728'), ('2019', '#1f77b4')]:
                if period in region_violence.columns:
                    high_violence = region_violence[period].get('High', 0)
                    fig.add_trace(
                        go.Bar(x=regions, y=high_violence[:15], name=f'{period} high violence %',
                              marker_color=color, opacity=0.7),
                        row=2, col=1
                    )
        
        fig.update_layout(height=800, title_text="Geographic Pattern Analysis: 2011 vs 2019")
        fig.show()
    
    # ===== 3. VIOLENCE PATTERNS =====
    
    def analyze_violence_patterns(self):
        """Deep dive into violence characteristics"""
        
        print("\n" + "="*80)
        print("VIOLENCE PATTERN ANALYSIS")
        print("="*80)
        
        # 3.1 Violence composition
        violence_comp = pd.crosstab(
            self.df_combined['period'],
            self.df_combined['violence_level'],
            normalize='index'
        ) * 100
        
        print("\n1. Violence Level Distribution (%):")
        print(violence_comp.round(2))
        
        # 3.2 Detailed sub-event comparison
        print("\n2. Sub-event Type Evolution:")
        for period, df in [('2011', self.df_2011), ('2019', self.df_2019)]:
            print(f"\n{period} - Top 10 Sub-event Types:")
            print(df['sub_event_type'].value_counts().head(10))
        
        # 3.3 Actor analysis (if available)
        actor_cols = [col for col in self.df_combined.columns if 'actor' in col.lower()]
        if actor_cols:
            for col in actor_cols:
                print(f"\n{col} - Top Actors:")
                for period, df in [('2011', self.df_2011), ('2019', self.df_2019)]:
                    print(f"\n{period}:")
                    print(df[col].value_counts().head(10))
        
        # 3.4 Violence escalation patterns
        def violence_escalation(df):
            df_sorted = df.sort_values('event_date')
            df_sorted['days_since'] = (df_sorted['event_date'] - df_sorted['event_date'].min()).dt.days
            df_sorted['violence_score'] = df_sorted['violence_level'].map({
                'Low/Peaceful': 1, 'Medium': 2, 'High': 3, 'Other': 1
            })
            
            # Rolling average of violence
            df_sorted['rolling_violence'] = df_sorted['violence_score'].rolling(window=7).mean()
            return df_sorted
        
        for period, df in [('2011', self.df_2011), ('2019', self.df_2019)]:
            esc = violence_escalation(df)
            peak_violence = esc[esc['rolling_violence'] == esc['rolling_violence'].max()]
            if len(peak_violence) > 0:
                print(f"\n{period} - Peak Violence Period: {peak_violence['event_date'].iloc[0].strftime('%Y-%m-%d')}")
        
        # Visualize violence
        fig = make_subplots(
            rows=2, cols=2,
            subplot_titles=(
                'Violence Level Distribution',
                'Violence Intensity Over Time',
                'Actor Type Comparison (if available)',
                'Event Type Co-occurrence'
            ),
            specs=[[{'type': 'pie'}, {'type': 'scatter'}],
                   [{'type': 'bar'}, {'type': 'heatmap'}]]
        )
        
        # Pie charts
        for i, (period, df) in enumerate([('2011', self.df_2011), ('2019', self.df_2019)]):
            violence_counts = df['violence_level'].value_counts()
            fig.add_trace(
                go.Pie(labels=violence_counts.index, values=violence_counts.values,
                      name=f'{period}', domain={'column': 0 if i==0 else 1}),
                row=1, col=1
            )
        
        # Violence intensity timeline
        for period, df, color in [('2011', self.df_2011, '#d62728'), ('2019', self.df_2019, '#1f77b4')]:
            daily_intensity = df.groupby('event_date')['violence_level'].apply(
                lambda x: (x == 'High').sum() / len(x) * 100
            )
            fig.add_trace(
                go.Scatter(x=daily_intensity.index, y=daily_intensity.values,
                          name=f'{period} % high violence', line=dict(color=color)),
                row=1, col=2
            )
        
        fig.update_layout(height=800, title_text="Violence Pattern Analysis: 2011 vs 2019")
        fig.show()
    
    # ===== 4. INTERSECTIONAL ANALYSIS =====
    
    def analyze_intersections(self):
        """Cross-analyze time, geography, and violence"""
        
        print("\n" + "="*80)
        print("INTERSECTIONAL ANALYSIS")
        print("="*80)
        
        # 4.1 Violence by time of day pattern
        # (If your data has time information)
        
        # 4.2 Geographic diffusion of violence
        if 'latitude' in self.df_combined.columns and 'longitude' in self.df_combined.columns:
            for period, df in [('2011', self.df_2011), ('2019', self.df_2019)]:
                high_violence = df[df['violence_level'] == 'High']
                print(f"\n{period} - High Violence Center:")
                if len(high_violence) > 0:
                    center_lat = high_violence['latitude'].mean()
                    center_lon = high_violence['longitude'].mean()
                    spread = high_violence[['latitude', 'longitude']].std()
                    print(f"Center: ({center_lat:.2f}, {center_lon:.2f})")
                    print(f"Spatial spread (std): lat={spread['latitude']:.2f}, lon={spread['longitude']:.2f}")
        
        # 4.3 Month-violence interaction
        month_violence = pd.crosstab(
            [self.df_combined['month'], self.df_combined['period']],
            self.df_combined['violence_level'],
            normalize='index'
        ) * 100
        
        print("\nSeasonal Violence Patterns:")
        for period in ['2011', '2019']:
            print(f"\n{period} - High violence months:")
            period_data = month_violence.xs(period, level=1)
            print(period_data[period_data['High'] > period_data['High'].mean()])
        
        # Visualization
        fig = make_subplots(
            rows=2, cols=2,
            subplot_titles=(
                'Violence Heatmap by Month',
                'Geographic Violence Shift',
                'Timeline-Event Type Interaction',
                'Violence Diffusion Pattern'
            )
        )
        
        # Violence heatmap
        heatmap_data = month_violence['High'].unstack(level=1)
        fig.add_trace(
            go.Heatmap(
                z=[heatmap_data['2011'].values, heatmap_data['2019'].values],
                x=heatmap_data.index,
                y=['2011', '2019'],
                colorscale='Reds'
            ),
            row=1, col=1
        )
        
        fig.update_layout(height=800, title_text="Intersectional Analysis")
        fig.show()
    
    # ===== 5. STATISTICAL COMPARISONS =====
    
    def statistical_tests(self):
        """Formal statistical comparisons"""
        
        print("\n" + "="*80)
        print("STATISTICAL ANALYSIS")
        print("="*80)
        
        # 5.1 Chi-square test for violence levels
        violence_table = pd.crosstab(
            self.df_combined['period'],
            self.df_combined['violence_level']
        )
        chi2, p_val, dof, expected = stats.chi2_contingency(violence_table)
        
        print(f"\n1. Violence Level Distribution Test:")
        print(f"Chi-square: {chi2:.2f}")
        print(f"P-value: {p_val:.4f}")
        print(f"Significant difference: {'Yes' if p_val < 0.05 else 'No'}")
        
        # 5.2 T-test for daily event counts
        daily_2011 = self.df_2011.groupby('event_date').size()
        daily_2019 = self.df_2019.groupby('event_date').size()
        
        t_stat, p_val = stats.ttest_ind(daily_2011, daily_2019)
        
        print(f"\n2. Daily Event Count Comparison:")
        print(f"2011 - Mean: {daily_2011.mean():.1f}, Std: {daily_2011.std():.1f}")
        print(f"2019 - Mean: {daily_2019.mean():.1f}, Std: {daily_2019.std():.1f}")
        print(f"T-statistic: {t_stat:.2f}")
        print(f"P-value: {p_val:.4f}")
        
        # 5.3 Mann-Whitney U test for violence intensity
        def get_daily_violence(df):
            return df.groupby('event_date')['violence_level'].apply(
                lambda x: (x == 'High').sum() / len(x) * 100 if len(x) > 0 else 0
            )
        
        violence_2011 = get_daily_violence(self.df_2011)
        violence_2019 = get_daily_violence(self.df_2019)
        
        u_stat, p_val = stats.mannwhitneyu(violence_2011, violence_2019, alternative='two-sided')
        
        print(f"\n3. Daily Violence Intensity:")
        print(f"2011 - Mean high violence %: {violence_2011.mean():.1f}%")
        print(f"2019 - Mean high violence %: {violence_2019.mean():.1f}%")
        print(f"Mann-Whitney U: {u_stat:.1f}")
        print(f"P-value: {p_val:.4f}")
        
        # 5.4 Effect size
        cohens_d = (violence_2011.mean() - violence_2019.mean()) / \
                   np.sqrt((violence_2011.var() + violence_2019.var()) / 2)
        
        print(f"\n4. Effect Size (Cohen's d): {cohens_d:.2f}")
        print(f"Interpretation: {'Large' if abs(cohens_d) > 0.8 else 'Medium' if abs(cohens_d) > 0.5 else 'Small'} effect")
    
    # ===== RUN FULL ANALYSIS =====
    
    def run_complete_analysis(self):
        """Execute all analyses"""
        
        print("="*80)
        print("COMPREHENSIVE COMPARISON: EGYPT PROTESTS 2011 vs 2019")
        print("="*80)
        
        self.analyze_temporal_patterns()
        self.analyze_geographic_patterns()
        self.analyze_violence_patterns()
        self.analyze_intersections()
        self.statistical_tests()
        
        # Key findings summary
        print("\n" + "="*80)
        print("KEY FINDINGS SUMMARY")
        print("="*80)
        
        # Calculate key metrics
        metrics = {
            'Total Events': [len(self.df_2011), len(self.df_2019)],
            'Unique Locations': [self.df_2011['location'].nunique(), self.df_2019['location'].nunique()],
            'High Violence %': [
                (self.df_2011['violence_level'] == 'High').mean() * 100,
                (self.df_2019['violence_level'] == 'High').mean() * 100
            ],
            'Peaceful %': [
                (self.df_2011['violence_level'] == 'Low/Peaceful').mean() * 100,
                (self.df_2019['violence_level'] == 'Low/Peaceful').mean() * 100
            ],
        }
        
        summary_df = pd.DataFrame(metrics, index=['2011', '2019']).T
        print("\nSummary Metrics:")
        print(summary_df.round(2))
        
        return summary_df

# Run the analysis
comparison = EgyptProtestsComparison(df_egypt2011_clean, df_alsisi_clean)
results = comparison.run_complete_analysis()

COMPREHENSIVE COMPARISON: EGYPT PROTESTS 2011 vs 2019

TEMPORAL PATTERN ANALYSIS

1. Peak Protest Days:

2011 - Top 5 Most Active Days:
event_date
2013-07-26    31
2011-01-25    29
2011-02-11    29
2013-08-14    28
2013-06-30    26
dtype: int64

2019 - Top 5 Most Active Days:
event_date
2020-09-25    14
2020-09-21    12
2020-09-20     9
2020-09-22     9
2019-09-27     8
dtype: int64

2. Day of Week Distribution (%):
period        2011   2019
day_of_week              
Friday       26.47  23.81
Monday       11.58  18.37
Saturday     10.55   8.16
Sunday       15.07  14.29
Thursday      8.27   8.84
Tuesday      14.29  16.33
Wednesday    13.78  10.20

3. Weekend vs Weekday Violence (%):
period           2011                      2019                    
violence_level   High Low/Peaceful Medium  High Low/Peaceful Medium
is_weekend                                                         
False           62.21        62.69  71.05  80.0        66.23  67.27
True            37.79        37.31  2


GEOGRAPHIC PATTERN ANALYSIS

2011 - Top 10 Protest Locations:
location
Cairo                   366
Cairo - Qasr al Nile    318
Cairo - Nasr City 1      94
Suez                     82
Alexandria               76
Sidi Jabir               63
Port Said                58
Al Mahallah al Kubra     50
Cairo - Bulaq            47
Al Arish                 44
Name: count, dtype: int64

2019 - Top 10 Protest Locations:
location
6th October City        13
Atfih                    7
10th of Ramadan City     6
Kafr ad Dawwar           5
Luxor                    5
Cairo - Nasr City 1      5
Minya                    4
Al Burullus              4
Aswan                    4
Al Ayyat                 4
Name: count, dtype: int64

Geographic Concentration (HHI):
2011: 0.0542
2019: 0.0262
Higher HHI = more concentrated in fewer locations

Violence by Region (%):
period           2011                      2019                    
violence_level   High Low/Peaceful Medium  High Low/Peaceful Medium
admin1       


VIOLENCE PATTERN ANALYSIS

1. Violence Level Distribution (%):
violence_level   High  Low/Peaceful  Medium
period                                     
2011            25.74         69.35    4.91
2019            10.20         52.38   37.41

2. Sub-event Type Evolution:

2011 - Top 10 Sub-event Types:
sub_event_type
Peaceful protest                      1611
Violent demonstration                  543
Protest with intervention              114
Excessive force against protesters      55
Name: count, dtype: int64

2019 - Top 10 Sub-event Types:
sub_event_type
Peaceful protest                      77
Protest with intervention             55
Violent demonstration                 10
Excessive force against protesters     5
Name: count, dtype: int64

actor1 - Top Actors:

2011:
actor1
Protesters (Egypt)                      1731
Rioters (Egypt)                          537
Police Forces of Egypt (2012-2013)        16
Unidentified Armed Group (Egypt)          11
Military Forces of Egypt (2011-2


INTERSECTIONAL ANALYSIS

2011 - High Violence Center:
Center: (29.99, 31.38)
Spatial spread (std): lat=1.30, lon=0.91

2019 - High Violence Center:
Center: (29.82, 31.17)
Spatial spread (std): lat=1.32, lon=0.77

Seasonal Violence Patterns:

2011 - High violence months:
violence_level       High  Low/Peaceful    Medium
month                                            
1               62.650602     28.313253  9.036145
3               36.458333     60.416667  3.125000
8               32.558140     63.565891  3.875969
11              34.507042     61.267606  4.225352

2019 - High violence months:
violence_level       High  Low/Peaceful     Medium
month                                             
1               33.333333     66.666667   0.000000
3               12.500000     50.000000  37.500000
7               25.000000     75.000000   0.000000
9               11.842105     28.947368  59.210526
12              16.666667     50.000000  33.333333



STATISTICAL ANALYSIS

1. Violence Level Distribution Test:
Chi-square: 232.80
P-value: 0.0000
Significant difference: Yes

2. Daily Event Count Comparison:
2011 - Mean: 4.0, Std: 4.5
2019 - Mean: 2.0, Std: 2.6
T-statistic: 3.72
P-value: 0.0002

3. Daily Violence Intensity:
2011 - Mean high violence %: 25.3%
2019 - Mean high violence %: 10.9%
Mann-Whitney U: 28403.0
P-value: 0.0000

4. Effect Size (Cohen's d): 0.46
Interpretation: Small effect

KEY FINDINGS SUMMARY

Summary Metrics:
                     2011    2019
Total Events      2323.00  147.00
Unique Locations   200.00   70.00
High Violence %     25.74   10.20
Peaceful %          69.35   52.38
